# 01 - Dashboard

All factors into one action.


## Setup

In [ ]:
# --- setup --------------------------------------------------------
import sys, json
sys.path.insert(0, r"D:\\Projects\\done\\eth_bot\\eth_bot_v2")
from pathlib import Path
from datetime import datetime, timezone
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import sqlite3

pd.set_option("display.width", 180)
pd.set_option("display.max_columns", 60)
plt.rcParams["figure.figsize"] = (13, 4)
plt.rcParams["axes.grid"] = True
plt.rcParams["grid.alpha"] = 0.3
np.random.seed(42)

SYMBOL = "ETHUSDT"
OUT = Path(r"D:\\Projects\\done\\eth_bot\\eth_bot_v2") / "reports" / "forecasts"
OUT.mkdir(parents=True, exist_ok=True)
DB = Path(r"D:\\Projects\\done\\eth_bot\\eth_bot_v2") / "database" / "market.db"

def sql(q, **p):
    with sqlite3.connect(DB) as c:
        return pd.read_sql_query(q, c, params=p)

from database.config.api.binance import get_klines

def fetch(symbol, interval, limit=500):
    raw = get_klines(symbol, interval=interval, limit=limit)
    df = pd.DataFrame(raw, columns=["open_time","open","high","low","close",
        "volume","close_time","quote_volume","trades","taker_base","taker_quote","ignore"])
    df["date"] = pd.to_datetime(df["open_time"], unit="ms", utc=True).dt.tz_convert(None)
    for c in ("open","high","low","close","volume"):
        df[c] = df[c].astype(float)
    return df[["date","open","high","low","close","volume"]].set_index("date").iloc[:-1]

def action_block(name, action, reason, price=None, entry=None, stop=None,
                 target=None, conf=None, extra=None):
    print("=" * 62)
    print(f"  {name}")
    print("=" * 62)
    print(f"  time   : {datetime.now(timezone.utc).strftime('%Y-%m-%d %H:%M')} UTC")
    if price is not None: print(f"  price  : {price:,.2f}")
    if conf  is not None: print(f"  conf   : {conf:.2f}")
    print(f"  ACTION : {action}")
    print(f"  REASON : {reason}")
    if entry is not None:
        print(f"  entry  : {entry:,.2f}   stop: {stop:,.2f}   target: {target:,.2f}")
    if extra:
        for k, v in extra.items():
            print(f"  {k:<6} : {v}")
    print("=" * 62)
    return {"action": action, "reason": reason, "price": price,
            "entry": entry, "stop": stop, "target": target, "conf": conf}

## 1. Fetch ETH 1h and 1d

In [ ]:
eth_1h = fetch("ETHUSDT", "1h", 500)
eth_1d = fetch("ETHUSDT", "1d", 500)
print("1h:", len(eth_1h), " 1d:", len(eth_1d))

1h: 499  1d: 499


## 2. Simple trend + momentum

In [ ]:
from analysis.indicators import add_all
d = add_all(eth_1d).dropna()
last = d.iloc[-1]
print("close:", last.close, " sma_50:", round(last.sma_50,2),
      " sma_200:", round(last.sma_200,2), " rsi:", round(last.rsi_14,2))

close: 2687.23  sma_50: 2484.66  sma_200: 2116.87  rsi: 57.79


## 3. Score each factor

In [ ]:
score = 0
if last.close > last.sma_50: score += 1
if last.sma_50 > last.sma_200: score += 1
if last.rsi_14 < 30: score += 1      # oversold, buy signal
if last.rsi_14 > 70: score -= 1      # overbought
if last.macd_hist > 0: score += 1
if last.macd_hist < 0: score -= 1

print("composite score:", score, "(range -3 to +4)")

composite score: 1 (range -3 to +4)


## 4. Final action

In [ ]:
if score >= 3:
    action, reason = "LONG", "strong bullish alignment"
elif score <= -2:
    action, reason = "SHORT", "strong bearish alignment"
else:
    action, reason = "FLAT", "no clear edge"

sig = action_block("ETHUSDT DASHBOARD", action, reason,
    price=float(last.close), conf=min(1.0, abs(score)/4),
    extra={"score": score})

(OUT/"signal_dashboard.json").write_text(json.dumps(sig, indent=2, default=str))
print("wrote signal_dashboard.json")

  ETHUSDT DASHBOARD
  time   : 2026-10-04 10:39 UTC
  price  : 2,687.23
  conf   : 0.25
  ACTION : FLAT
  REASON : no clear edge
  score  : 1
wrote signal_dashboard.json
